# FaceFlash — reproduce the benchmarks yourself

People said the numbers look AI-generated. Fair enough. Run this notebook and see for yourself.

What this does:
1. Pulls **real MS1MV2 ArcFace embeddings** from a public HuggingFace dataset (no token needed)
2. Builds FaceFlash, FAISS-Flat, and HNSWLIB indexes on the same 100K embeddings
3. Measures recall@1, memory, and latency — with charts

Runtime: ~5 min on free Colab CPU. No GPU required.


In [ ]:
import os, subprocess

# Clone the repo
if not os.path.exists('/content/faceflash'):
    !git clone --depth 1 https://github.com/raghavenderreddygrudhanti/faceflash.git /content/faceflash

os.chdir('/content/faceflash')

# Install deps
!pip install -q numpy faiss-cpu hnswlib huggingface_hub matplotlib

# Try to build Rust backend (makes search faster, but not required for correctness)
try:
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y 2>&1 | tail -1
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
    !pip install -q maturin
    !maturin develop --release 2>&1 | tail -3
    from faceflash import _core
    print('Rust backend loaded (SIMD-accelerated search)')
except Exception as e:
    print(f'Rust backend not available ({e}) — using numpy fallback (same recall, slower)')

print('Setup complete.')


## Download MS1MV2 embeddings

Real ArcFace embeddings from MS-Celeb-1M (cleaned). Public HuggingFace, no token.


In [ ]:
import os, sys
import numpy as np
os.chdir("/content/faceflash")
sys.path.insert(0, "/content/faceflash")
from huggingface_hub import hf_hub_download

REPO = "raghavenderreddy1212/faceflash-embeddings"
os.makedirs("data", exist_ok=True)
print("Downloading MS1MV2 embeddings...")
hf_hub_download(REPO, "ms1m_embeddings.npy", repo_type="dataset", local_dir="data")
hf_hub_download(REPO, "ms1m_labels.npy", repo_type="dataset", local_dir="data")

embeddings = np.load("data/ms1m_embeddings.npy")
labels = np.load("data/ms1m_labels.npy")
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings = (embeddings / norms).astype(np.float32)
print(f"Loaded: {embeddings.shape[0]:,} embeddings, {len(np.unique(labels)):,} identities")
print(f"Raw float size: {embeddings.nbytes / 1e6:.0f} MB")


## Setup: 100K database, 1000 queries


In [ ]:
N_DB = 100_000
N_QUERIES = 1000
rng = np.random.default_rng(42)
perm = rng.permutation(len(embeddings))
db = embeddings[perm[:N_DB]]
queries = embeddings[perm[N_DB:N_DB+N_QUERIES]]
print(f"Database: {N_DB:,} faces ({db.nbytes/1e6:.1f} MB)")
print(f"Queries: {N_QUERIES:,}")


## Ground truth: FAISS-Flat exact cosine


In [ ]:
import faiss, time
index_flat = faiss.IndexFlatIP(512)
index_flat.add(db)
t0 = time.perf_counter()
_, gt_ids = index_flat.search(queries, 1)
faiss_flat_time = time.perf_counter() - t0
gt_ids = gt_ids.flatten()
faiss_flat_mem_mb = db.nbytes / 1e6
print(f"FAISS-Flat: {N_QUERIES} queries in {faiss_flat_time*1000:.0f}ms")
print(f"Memory: {faiss_flat_mem_mb:.1f} MB")


## FaceFlash: PCA+ITQ binary + cosine rerank


In [ ]:
from faceflash.pca_quantize import PCABinaryQuantizer, _POPCOUNT_TABLE

# Try Rust backend, fall back to numpy popcount (same recall, just slower)
try:
    from faceflash import _core as rust_core
    HAS_RUST = True
    print('Using Rust SIMD kernel')
except ImportError:
    HAS_RUST = False
    print('Using numpy fallback (same recall, ~5x slower)')

quantizer = PCABinaryQuantizer(n_bits=512)
quantizer.fit(db)
db_codes = quantizer.transform(db)
query_codes = quantizer.transform(queries)
ff_mem_mb = db_codes.nbytes / 1e6
print(f"FaceFlash index: {ff_mem_mb:.2f} MB ({db.nbytes//db_codes.nbytes}x compression)")

N_CANDIDATES = 100
ff_latencies = []
ff_predicted = np.empty(N_QUERIES, dtype=np.int64)

def hamming_topk_numpy(query_code, db_codes, k):
    """Numpy fallback: same result as the Rust kernel, just slower."""
    xor = np.bitwise_xor(db_codes, query_code)
    dists = _POPCOUNT_TABLE[xor].sum(axis=1)
    top_k = np.argpartition(dists, k)[:k]
    return top_k

for i in range(N_QUERIES):
    t0 = time.perf_counter()
    if HAS_RUST:
        top_idx = rust_core.hamming_topk(query_codes[i].tobytes(), db_codes, N_CANDIDATES)
    else:
        top_idx = hamming_topk_numpy(query_codes[i], db_codes, N_CANDIDATES)
    sims = db[top_idx] @ queries[i]
    ff_predicted[i] = top_idx[np.argmax(sims)]
    ff_latencies.append(time.perf_counter() - t0)

ff_recall = (ff_predicted == gt_ids).mean() * 100
ff_avg_ms = np.mean(ff_latencies) * 1000
print(f"\nRecall@1: {ff_recall:.1f}%")
print(f"Avg latency: {ff_avg_ms:.2f}ms")


## HNSWLIB comparison


In [ ]:
import hnswlib
hnsw = hnswlib.Index(space='ip', dim=512)
hnsw.init_index(max_elements=N_DB, ef_construction=200, M=16)
hnsw.add_items(db, np.arange(N_DB))
hnsw.set_ef(128)

hnsw_latencies = []
hnsw_predicted = np.empty(N_QUERIES, dtype=np.int64)
for i in range(N_QUERIES):
    t0 = time.perf_counter()
    ids, _ = hnsw.knn_query(queries[i:i+1], k=1)
    hnsw_predicted[i] = ids[0][0]
    hnsw_latencies.append(time.perf_counter() - t0)

hnsw_recall = (hnsw_predicted == gt_ids).mean() * 100
hnsw_avg_ms = np.mean(hnsw_latencies) * 1000
hnsw_mem_mb = db.nbytes * 1.5 / 1e6
print(f"HNSWLIB: recall@1={hnsw_recall:.1f}%, latency={hnsw_avg_ms:.2f}ms, mem=~{hnsw_mem_mb:.0f}MB")


## Isolation: same codes through FAISS IndexBinaryFlat

Proves the kernel is not special — same codes, different library, same recall.


In [ ]:
bin_index = faiss.IndexBinaryFlat(512)
bin_index.add(db_codes)
faiss_bin_predicted = np.empty(N_QUERIES, dtype=np.int64)
for i in range(N_QUERIES):
    _, I = bin_index.search(query_codes[i:i+1], N_CANDIDATES)
    cand = I[0][I[0] >= 0]
    sims = db[cand] @ queries[i]
    faiss_bin_predicted[i] = cand[np.argmax(sims)]
faiss_bin_recall = (faiss_bin_predicted == gt_ids).mean() * 100
print(f"FaceFlash kernel: {ff_recall:.1f}%")
print(f"FAISS BinaryFlat: {faiss_bin_recall:.1f}%")
print(f"Match: {abs(ff_recall - faiss_bin_recall) < 0.5}")
print()
print("Same recall = the kernel is not doing anything special.")
print("The accuracy comes from PCA+ITQ preserving neighbor ordering.")


## Charts


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
methods = ["FAISS-Flat\n(exact)", "FaceFlash\n(512-bit)", "HNSWLIB\n(ef=128)"]
colors = ["#4e79a7", "#59a14f", "#e15759"]

mem_vals = [faiss_flat_mem_mb, ff_mem_mb, hnsw_mem_mb]
bars = axes[0].bar(methods, mem_vals, color=colors, edgecolor="black", linewidth=0.4)
axes[0].set_ylabel("Index Memory (MB)")
axes[0].set_title(f"Memory @ {N_DB//1000}K faces")
axes[0].bar_label(bars, fmt="%.1f MB", fontsize=9)
axes[0].set_ylim(0, max(mem_vals)*1.3)

recall_vals = [100.0, ff_recall, hnsw_recall]
bars = axes[1].bar(methods, recall_vals, color=colors, edgecolor="black", linewidth=0.4)
axes[1].set_ylabel("Recall@1 (%)")
axes[1].set_title("Recall@1")
axes[1].set_ylim(95, 101.5)
axes[1].bar_label(bars, fmt="%.1f%%", fontsize=9)
axes[1].axhline(100, color="gray", linestyle="--", alpha=0.3)

lat_vals = [faiss_flat_time/N_QUERIES*1000, ff_avg_ms, hnsw_avg_ms]
bars = axes[2].bar(methods, lat_vals, color=colors, edgecolor="black", linewidth=0.4)
axes[2].set_ylabel("Avg Latency (ms)")
axes[2].set_title("Single-query Latency")
axes[2].bar_label(bars, fmt="%.2f ms", fontsize=9)

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
plt.suptitle(f"FaceFlash vs Competitors - {N_DB//1000}K Real MS1MV2 Faces", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Memory scaling projection
fig, ax = plt.subplots(figsize=(9, 5))
scales = [100_000, 200_000, 500_000, 1_000_000]
slabels = ["100K","200K","500K","1M"]
fbpf = db_codes.shape[1]
fpf = 512*4
hpf = int(fpf*1.5)
ff_m = [s*fbpf/1e6 for s in scales]
fl_m = [s*fpf/1e6 for s in scales]
hn_m = [s*hpf/1e6 for s in scales]
x = np.arange(4)
w = 0.25
ax.bar(x-w, fl_m, w, label="FAISS-Flat", color="#4e79a7")
ax.bar(x, hn_m, w, label="HNSWLIB", color="#e15759")
ax.bar(x+w, ff_m, w, label="FaceFlash", color="#59a14f")
ax.set_xticks(x)
ax.set_xticklabels(slabels)
ax.set_xlabel("Number of Faces")
ax.set_ylabel("Index Memory (MB)")
ax.set_title("Memory Scaling (same recall)")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
for i,v in enumerate(ff_m):
    ax.annotate(f"{v:.0f}MB",(x[i]+w,v),ha="center",va="bottom",fontsize=8)
plt.tight_layout()
plt.show()
print(f"At 1M: FaceFlash={ff_m[-1]:.0f}MB, HNSW=~{hn_m[-1]:.0f}MB, FAISS={fl_m[-1]:.0f}MB")


## Final report


In [ ]:
print("="*65)
print("  BENCHMARK REPORT")
print("="*65)
print(f"  Dataset:    MS1MV2 (public HuggingFace)")
print(f"  Scale:      {N_DB:,} database, {N_QUERIES:,} queries")
print(f"  Hardware:   Colab free CPU")
print()
print(f"  {'Method':<30}{'Recall@1':<10}{'Memory':<12}{'Latency'}")
print(f"  {'-'*60}")
print(f"  {'FAISS-Flat (exact)':<30}{'100.0%':<10}{faiss_flat_mem_mb:<10.1f}MB  {faiss_flat_time/N_QUERIES*1000:.2f}ms")
print(f"  {'FaceFlash (512-bit)':<30}{str(round(ff_recall,1))+'%':<10}{ff_mem_mb:<10.2f}MB  {ff_avg_ms:.2f}ms")
print(f"  {'HNSWLIB (ef=128)':<30}{str(round(hnsw_recall,1))+'%':<10}~{hnsw_mem_mb:<9.0f}MB  {hnsw_avg_ms:.2f}ms")
print(f"  {'-'*60}")
print(f"  {'FAISS BinaryFlat (isolation)':<30}{str(round(faiss_bin_recall,1))+'%'}")
print()
print(f"  {db.nbytes//db_codes.nbytes}x less memory than exact, same recall.")
print(f"  {hnsw_mem_mb/ff_mem_mb:.0f}x less memory than HNSW.")
print(f"  Isolation: kernel is not special, compression is.")
print("="*65)


---

**What you just saw:** real embeddings, real competitors, real numbers.
Save with outputs (File > Save a copy in GitHub) to preserve the evidence.

Full 1M run:
```
git clone https://github.com/raghavenderreddygrudhanti/faceflash
cd faceflash && bash scripts/runpod_ms1m.sh
```
